# STAC data prep — Seabed lithology (single COG)

Notebook to migrate a regular 2D grid in `.tif` format to a CF compliant **Cloud Optimized GeoTIFF (CoG)**.
Same pattern as `11b_Salinity.ipynb`, but for a single static layer (no scenario/time loop).

- **Case:** 2D raster, one layer, categorical (13 lithology class codes), global 0.1° grid, `int16`, ~12 MB.
- **Format:** one COG (no tiling needed, see `DATA_FORMATS.md`).
- **Resampling:** `NEAREST` for overviews, because class codes must never be averaged.
- **Metadata template:** `metadata/metadata_Seabed_lithology.json` (Dutkiewicz et al. 2015 / PANGAEA). The empty/incorrect fields are refreshed from the raster below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\seabed_litho.tif`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1` from the host. It copies the input to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import rasterio
import rioxarray as rxr

### Define drive paths

`AFRICA_DATA` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "seabed_litho"
COLLECTION_ID = "seabed_lithology"

# Same layout as P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian (DATA_VIVIAN is set in docker,
# where the P: drive is not mounted; a local mirror of that folder is used and the stac/ output is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_tif = data_vivian / f"{VARIABLE}.tif"
metadata_template_path = stac_repo / "metadata" / "metadata_Seabed_lithology.json"

processed_data_dir = data_vivian / "stac" / VARIABLE
cog_dir = processed_data_dir / "cog"
cog_dir.mkdir(parents=True, exist_ok=True)

for p in (source_tif, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_tif)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: /workspace/africa_data/Data_Vivian/seabed_litho.tif
Metadata template: /workspace/global-coastal-atlas/STAC/data/metadata/metadata_Seabed_lithology.json
Output dir: /workspace/africa_data/Data_Vivian/stac/seabed_litho


### Read raw data

In [3]:
# mask_and_scale=False keeps the integer class codes (no float conversion)
ds = rxr.open_rasterio(source_tif, mask_and_scale=False)

with rasterio.open(source_tif) as src:
    print("dtype:", src.dtypes[0], "| nodata:", src.nodata, "| size:", src.width, "x", src.height)
    print("bounds:", src.bounds, "| res:", src.res, "| tiled:", src.is_tiled, "| overviews:", src.overviews(1))

classes = np.unique(ds.values)
classes = classes[classes != ds.rio.nodata]
print("Lithology class codes:", classes.tolist())
ds

dtype: int16 | nodata: -32768.0 | size: 3601 x 1801
bounds: BoundingBox(left=-180.05, bottom=-90.05, right=180.05, top=90.05) | res: (0.1, 0.09999999999999999) | tiled: False | overviews: []


Lithology class codes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


<xarray.DataArray (band: 1, y: 1801, x: 3601)> Size: 13MB
array([[[-32768, -32768, ..., -32768, -32768],
        [-32768, -32768, ..., -32768, -32768],
        ...,
        [-32768, -32768, ..., -32768, -32768],
        [-32768, -32768, ..., -32768, -32768]]],
      shape=(1, 1801, 3601), dtype=int16)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 14kB 90.0 89.9 89.8 89.7 ... -89.8 -89.9 -90.0
  * x            (x) float64 29kB -180.0 -179.9 -179.8 ... 179.8 179.9 180.0
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     -32768
    scale_factor:   1.0
    add_offset:     0.0

### Check CF compliancy original file

Not applicable for a plain GeoTIFF (same approach as `11b_Salinity`): CRS, nodata and attributes are written explicitly below.

### Make CF compliant alterations (dataset dependent)

Fill the metadata fields that are empty (or wrong) in the template using the raster itself:
`SPATIAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `DATA_MODEL`, `COLLECTION_ID`, `UNITS`, ...

In [4]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Process data
da = ds.isel(band=0).drop_vars("band")
da = da.rio.write_crs(4326)
nodata = int(ds.rio.nodata)
da = da.rio.write_nodata(nodata)
da.attrs["_FillValue"] = nodata
da.name = VARIABLE

# Bounds are pixel edges (-180.05 ... 180.05); STAC bbox must stay inside the valid EPSG:4326 range
left, bottom, right, top = da.rio.bounds()
bbox = [max(left, -180.0), max(bottom, -90.0), min(right, 180.0), min(top, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "SeabedLithology",
    "HISTORY": [
        "Dutkiewicz et al. (2015) seafloor lithology map, published on PANGAEA (2021)",
        "Gridded 0.1 degree GeoTIFF (seabed_litho.tif), class codes stored as int16",
        "Converted to a single Cloud Optimized GeoTIFF (DEFLATE, nearest-neighbour overviews)",
    ],
    "MEDIA_TYPE": "image/tiff; application=geotiff; profile=cloud-optimized",
    "DATA_MODEL": "raster_cog",
    "DIMENSIONS": ["x", "y"],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    "TEMPORAL_EXTENT": ["", ""],
    "KEYWORDS": ["seabed", "seafloor sediments", "lithology", "geology", "ocean", "global"],
    "TAGS": ["seabed", "lithology", "sediment", "cog"],
    "LONG_NAME": "Seafloor lithology class",
    "UNITS": "class code (categorical)",
    "COMMENT": (
        "Categorical raster: each pixel holds an integer lithology class code "
        f"({', '.join(str(c) for c in classes.tolist())}); {nodata} is nodata (land / no data). "
        "Values must not be interpolated or averaged, overviews use nearest-neighbour resampling. "
        "Class code labels follow the colour table (cpt) published with the PANGAEA dataset."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(abs(da.rio.resolution()[0]), 6),
    "NODATA": nodata,
    "DATA_TYPE": str(da.dtype),
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": COLLECTION_ID,  # GeoServer workspace used by scripts/17_seabed_litho.ipynb
})

# Add attributes to the dataset
for attr_name, attr_val in metadata.items():
    if attr_name == "PROVIDERS":
        attr_val = json.dumps(attr_val)  # include a dictionary as attribute
    if attr_name == "MEDIA_TYPE":
        attr_val = "IMAGE/TIFF"  # same convention as 11b_Salinity
    if isinstance(attr_val, list):
        attr_val = json.dumps(attr_val)  # GeoTIFF tags only hold scalars
    da.attrs[attr_name] = attr_val
da.attrs["Conventions"] = "CF-1.8"

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("NODATA:", metadata["NODATA"], "| DATA_TYPE:", metadata["DATA_TYPE"])

SPATIAL_EXTENT: [-180.0, -90.0, 180.0, 90.0]
SPATIAL_RESOLUTION: 0.1
NODATA: -32768 | DATA_TYPE: int16


### Write data to CoG

Single CoG, `DEFLATE` compression, **nearest-neighbour** overviews because the data are class codes.

In [5]:
out_path = cog_dir / f"{VARIABLE}.tif"
da.rio.to_raster(
    out_path,
    driver="COG",
    compress="DEFLATE",
    dtype="int16",
    overview_resampling="NEAREST",
    resampling="NEAREST",
)

metadata_output_path = processed_data_dir / f"metadata_{VARIABLE}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Wrote COG:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote COG: /workspace/africa_data/Data_Vivian/stac/seabed_litho/cog/seabed_litho.tif (0.3 MB)
Wrote metadata: /workspace/africa_data/Data_Vivian/stac/seabed_litho/metadata_seabed_litho.json


### Verify the output

In [6]:
with rasterio.open(out_path) as src:
    assert src.dtypes[0] == "int16"
    assert src.nodata == nodata
    assert src.crs.to_epsg() == 4326
    assert src.is_tiled, "COG must be internally tiled"
    assert len(src.overviews(1)) > 0, "COG must have overviews"
    new = src.read(1)
    print("Layout:", src.profile.get("blockxsize"), "x", src.profile.get("blockysize"), "| overviews:", src.overviews(1))
    print("Compression:", src.compression)

# Pixel values must be identical to the source (no class codes created or lost)
with rasterio.open(source_tif) as src:
    old = src.read(1)
assert np.array_equal(old, new), "Pixel values changed during COG conversion"
assert set(np.unique(new)) == set(np.unique(old))
print("OK: seabed_litho COG and metadata are ready for the STAC builder notebook.")

Layout: 512 x 512 | overviews: [2, 4, 8]
Compression: Compression.deflate


OK: seabed_litho COG and metadata are ready for the STAC builder notebook.
